# 04 · Calibração e explicabilidade

**Etapas 5 e 6 do pipeline.** Recebe o modelo vencedor e a base escorada (etapa 4).

| passo | onde roda | o que faz |
|---|---|---|
| 1 | Spark | ajusta a calibração no **teste** e cria `score_calibrado` em toda a base |
| 2 | Spark | tabela e gráfico de calibração (antes × depois) |
| 3 | Spark → pandas | sorteia a amostra do treino para a explicação |
| 4 | pandas | **espelho**: o modelo vencedor reproduzido em pandas, com o SHAP de cada contrato |
| 5 | pandas | teste de fidelidade (espelho × modelo Spark) |
| 6 | pandas | beeswarm do SHAP e gráfico de importância |

## Calibração

O modelo ordena bem os contratos, mas a probabilidade que ele devolve pode estar fora do nível. **Exemplo:** o modelo dá PD média de 7,4% no teste, e a taxa observada no teste foi 6,0%. A calibração corrige o nível sem mudar a ordem dos contratos.

| método | tarefa | fórmula | corrige |
|---|---|---|---|
| Platt | PD | PD' = 1 / (1 + e^−(a·logit(PD) + b)) | nível e inclinação (modelo confiante demais ou de menos) |
| isotônica | PD e LGD | ajuste em degraus que só sobe | qualquer distorção monotônica |
| linear | LGD | LGD' = a·LGD + b, limitada a [0, 1] | nível e inclinação |

O ajuste usa o **teste** (safras posteriores ao treino). O OOT só mede o resultado.

## Espelho e explicação

O **espelho** é o modelo vencedor reproduzido em pandas, para calcular o SHAP de cada contrato. SHAP (*SHapley Additive exPlanations*) reparte a previsão de cada contrato entre as variáveis: valor base + soma das contribuições = previsão.

Para todas as técnicas, o espelho é **o mesmo modelo** (não um retreino):

| técnica | como o espelho é obtido | escala do SHAP |
|---|---|---|
| logística, logística fracionária | coeficientes do modelo Spark; SHAP = β · (x − média de x) | logit |
| Random Forest, GBT | o pacote `shap` lê as árvores do modelo Spark diretamente | probabilidade (RF) ou margem (GBT) |
| XGBoost | o *booster* nativo do modelo (`get_booster()`) | logit (PD) / margem (LGD) |
| LightGBM | o modelo nativo exportado pelo SynapseML (`getNativeModel()`), carregado no `lightgbm` | logit (PD) / margem (LGD) |
| duas etapas | cada etapa pela sua técnica; os gráficos saem por etapa | — |

O **teste de fidelidade** confirma a equivalência na própria amostra: correlação de Spearman entre o score do Spark e o do espelho, percentual de contratos no mesmo decil e diferença no KS (PD) ou no RMSE (LGD).

O `03_automl` já executa o `00_config` e traz `aplica_plano` e `_escora_parte`, usados para preparar a amostra exatamente como o modelo a recebe. (No Databricks, o `%run` precisa ficar sozinho na célula.)

In [ ]:
%run ./03_automl

In [ ]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler

## 1. Calibração

In [ ]:
def _logit(coluna):
    """logit(p) = ln(p / (1 − p)), com p limitado a [1e-6, 1 − 1e-6] para evitar ±infinito."""
    p = F.least(F.greatest(coluna, F.lit(1e-6)), F.lit(1 - 1e-6))
    return F.log(p / (1 - p))


def ajusta_calibracao(base_escorada, cfg, coluna_score="score_bruto"):
    """
    Ajusta a calibração no teste e devolve os parâmetros (dicionário JSON).

    Conceito técnico:
        - Platt: regressão logística (Spark) com uma única variável, logit(score).
          Os parâmetros a (inclinação) e b (intercepto) bastam para aplicar.
        - Isotônica: IsotonicRegression do Spark; guardam-se os pontos de
          quebra e os valores, e a aplicação é uma interpolação linear entre eles.
        - Linear (LGD): mínimos quadrados de LGD observada sobre a prevista,
          calculados com somas (uma agregação).
    """
    metodo = cfg["calibracao"]["metodo"]
    alvo = cfg["colunas"]["alvo"]
    teste = base_escorada.filter(F.col("particao") == "teste").select(
        F.col(coluna_score).alias("s"), F.col(alvo).cast("double").alias("y"))

    if metodo == "platt":
        from pyspark.ml.classification import LogisticRegression
        dados = VectorAssembler(inputCols=["z"], outputCol="v").transform(teste.withColumn("z", _logit(F.col("s"))))
        modelo = LogisticRegression(featuresCol="v", labelCol="y", regParam=0.0,
                                    standardization=False, maxIter=200).fit(dados)
        return {"metodo": "platt", "a": float(modelo.coefficients[0]), "b": float(modelo.intercept)}

    if metodo == "isotonica":
        from pyspark.ml.regression import IsotonicRegression
        dados = VectorAssembler(inputCols=["s"], outputCol="v").transform(teste)
        modelo = IsotonicRegression(featuresCol="v", labelCol="y", isotonic=True).fit(dados)
        return {"metodo": "isotonica", "pontos": [float(v) for v in modelo.boundaries.toArray()],
                "valores": [float(v) for v in modelo.predictions.toArray()]}

    # linear (LGD): y = b0 + b1·s
    soma = teste.agg(F.count(F.lit(1)).alias("n"), F.sum("s").alias("s"), F.sum("y").alias("y"),
                     F.sum(F.col("s") * F.col("s")).alias("ss"), F.sum(F.col("s") * F.col("y")).alias("sy")).first()
    n = soma["n"]
    variancia = soma["ss"] - soma["s"] ** 2 / n
    covariancia = soma["sy"] - soma["s"] * soma["y"] / n
    b1 = covariancia / variancia if variancia > 0 else 1.0
    b0 = soma["y"] / n - b1 * soma["s"] / n
    return {"metodo": "linear", "a": float(b1), "b": float(b0)}


def aplica_calibracao(sdf, calibracao, coluna_entrada="score_bruto", coluna_saida="score_calibrado"):
    """Cria `coluna_saida` com o score calibrado (regra pura, sem objeto ajustado)."""
    s = F.col(coluna_entrada)
    if calibracao["metodo"] == "platt":
        z = F.lit(calibracao["a"]) * _logit(s) + F.lit(calibracao["b"])
        return sdf.withColumn(coluna_saida, 1 / (1 + F.exp(-z)))
    if calibracao["metodo"] == "linear":
        valor = F.lit(calibracao["b"]) + F.lit(calibracao["a"]) * s
        return sdf.withColumn(coluna_saida, F.least(F.greatest(valor, F.lit(0.0)), F.lit(1.0)))
    # isotônica: interpolação linear entre os pontos (np.interp), por partição
    from pyspark.sql.functions import pandas_udf
    pontos, valores = np.array(calibracao["pontos"]), np.array(calibracao["valores"])

    @pandas_udf("double")
    def interpola(serie: pd.Series) -> pd.Series:
        return pd.Series(np.interp(serie.to_numpy(dtype=float), pontos, valores))

    return sdf.withColumn(coluna_saida, interpola(s))

## 2. Tabela e gráfico de calibração

**Tabela**, por partição (treino, teste, OOT): média prevista antes e depois da calibração, média observada, e as métricas de calibração antes e depois (log loss e Brier na PD; RMSE e viés na LGD).

**Gráfico**, no OOT: os contratos são divididos em 10 faixas do score calibrado. Para cada faixa, um ponto com a média prevista (eixo x) contra a média observada (eixo y), antes e depois. Calibração perfeita = pontos sobre a diagonal.

In [ ]:
def tabela_calibracao(base_calibrada, cfg):
    """Antes × depois da calibração, por partição."""
    y = F.col(cfg["colunas"]["alvo"]).cast("double")
    antes, depois = F.col("score_bruto"), F.col("score_calibrado")
    agregacoes = [F.count(F.lit(1)).alias("contratos"), F.avg(y).alias("media_observada"),
                  F.avg(antes).alias("media_prevista_antes"), F.avg(depois).alias("media_prevista_depois")]
    if cfg["tarefa"] == "pd":
        def log_loss(p):
            p = F.least(F.greatest(p, F.lit(1e-12)), F.lit(1 - 1e-12))
            return F.avg(-(y * F.log(p) + (1 - y) * F.log(1 - p)))
        agregacoes += [log_loss(antes).alias("log_loss_antes"), log_loss(depois).alias("log_loss_depois"),
                       F.avg((antes - y) ** 2).alias("brier_antes"), F.avg((depois - y) ** 2).alias("brier_depois")]
    else:
        agregacoes += [F.sqrt(F.avg((antes - y) ** 2)).alias("rmse_antes"),
                       F.sqrt(F.avg((depois - y) ** 2)).alias("rmse_depois"),
                       F.avg(antes - y).alias("vies_antes"), F.avg(depois - y).alias("vies_depois")]
    tabela = (base_calibrada.filter(F.col("particao").isin("treino", "teste", "oot"))
              .groupBy("particao").agg(*agregacoes).toPandas())
    ordem = {"treino": 0, "teste": 1, "oot": 2}
    return tabela.sort_values("particao", key=lambda s: s.map(ordem)).reset_index(drop=True)


def grafico_calibracao(base_calibrada, cfg, particao="oot", n_faixas=10):
    """Previsto × observado por decil do score calibrado, antes e depois."""
    import matplotlib.pyplot as plt
    conjunto = base_calibrada.filter(F.col("particao") == particao)
    cortes = sorted(set(conjunto.approxQuantile("score_calibrado",
                                                [k / n_faixas for k in range(1, n_faixas)], 1e-4)))
    faixa = F.lit(len(cortes))
    for indice, corte in reversed(list(enumerate(cortes))):
        faixa = F.when(F.col("score_calibrado") <= F.lit(corte), F.lit(indice)).otherwise(faixa)
    t = (conjunto.groupBy(faixa.alias("faixa"))
         .agg(F.avg("score_bruto").alias("antes"), F.avg("score_calibrado").alias("depois"),
              F.avg(F.col(cfg["colunas"]["alvo"]).cast("double")).alias("observada"))
         .toPandas().sort_values("faixa"))
    figura, eixo = plt.subplots(figsize=(5.6, 5.0), dpi=130)
    maximo = float(np.nanmax(t[["antes", "depois", "observada"]].to_numpy())) * 1.08
    eixo.plot([0, maximo], [0, maximo], color="#b4b2a9", lw=1, ls=(0, (4, 3)), label="calibração perfeita")
    eixo.plot(t["antes"], t["observada"], marker="o", color="#eb6834", lw=1.5, label="antes (score bruto)")
    eixo.plot(t["depois"], t["observada"], marker="o", color="#2a78d6", lw=2, label="depois (calibrado)")
    nome = "PD" if cfg["tarefa"] == "pd" else "LGD"
    eixo.set_xlabel(f"{nome} prevista média da faixa")
    eixo.set_ylabel(f"{nome} observada média da faixa")
    eixo.set_xlim(0, maximo)
    eixo.set_ylim(0, maximo)
    eixo.spines[["top", "right"]].set_visible(False)
    eixo.grid(color="#e4e3df", lw=0.8)
    eixo.legend(frameon=False, fontsize=8, loc="upper left")
    eixo.set_title(f"Calibração no {particao.upper()} (10 faixas do score)", loc="left", fontsize=11)
    figura.tight_layout()
    return figura

## 3. Amostra para a explicação

Sorteada do **treino**, com `tamanho_amostra` contratos (50.000 por padrão). Para cada parte do modelo (uma, ou duas no modelo de duas etapas) a amostra passa pelo plano de pré-processamento daquela parte e vai para pandas com:

- as colunas de atributos `f_XXXX` (exatamente o que o modelo recebe);
- o score do modelo Spark naquela amostra (para o teste de fidelidade);
- as variáveis originais (para colorir o beeswarm);
- o alvo.

In [ ]:
def amostra_para_explicacao(base_calibrada, modelo, contexto, cfg):
    """
    Devolve {nome_da_parte: (parte, pandas)} com a amostra preparada de cada parte.

    No duas etapas, a "previsão Spark" de cada etapa é a saída daquela etapa
    (P(perda) na 1ª, LGD da etapa 2 na 2ª), escorada na mesma amostra.
    """
    alvo = cfg["colunas"]["alvo"]
    n = cfg["explicabilidade"]["tamanho_amostra"]
    treino = base_calibrada.filter(F.col("particao") == "treino")
    total = treino.count()
    amostra = (treino.sample(fraction=min(1.0, 1.2 * n / total), seed=cfg["semente"]).limit(n)
               .select(*contexto["variaveis"], alvo, "score_bruto").persist())
    if modelo["tecnica"] == "duas_etapas":
        partes = {"etapa1": modelo["etapa1"], "etapa2": modelo["etapa2"]}
    else:
        partes = {"modelo": modelo}
    resultado = {}
    for nome, parte in partes.items():
        if modelo["tecnica"] == "duas_etapas":
            escorada = _escora_parte(parte, amostra, "_score_spark")
        else:
            escorada = amostra.withColumn("_score_spark", F.col("score_bruto"))
        colunas_f = [a["coluna"] for a in parte["plano"]["atributos"]]
        preparada = aplica_plano(escorada, parte["plano"]).select(
            *colunas_f, "_score_spark", alvo, *[F.col(v).alias(f"orig__{v}") for v in contexto["variaveis"]])
        pdf = preparada.toPandas()
        pdf[colunas_f] = pdf[colunas_f].astype("float64")
        resultado[nome] = (parte, pdf)
    amostra.unpersist()
    return resultado

## 4. Espelho e SHAP

`espelho` devolve, para uma parte do modelo e a matriz de atributos em pandas:

- a matriz SHAP (contratos × atributos);
- o valor base (a previsão média, na escala do SHAP);
- a previsão do espelho (na escala do score, para comparar com o Spark);
- a importância nativa de cada atributo, lida do modelo Spark.

**Importância nativa por técnica:**

| técnica | medida |
|---|---|
| logística | \|β\| × desvio-padrão do atributo (efeito de uma variação típica) |
| Random Forest, GBT | `featureImportances` do Spark (redução de impureza) |
| XGBoost, LightGBM | ganho (quanto o atributo reduziu o erro nas árvores) |

In [ ]:
def _texto_modelo_lightgbm(ajustado):
    """
    Texto do modelo nativo do LightGBM a partir do modelo SynapseML.

    Conceito técnico:
        getNativeModel() devolve o modelo no formato texto do LightGBM. Se a
        versão do SynapseML não tiver esse método, saveNativeModel grava o mesmo
        texto num arquivo, que é lido de volta.
    """
    if hasattr(ajustado, "getNativeModel"):
        return ajustado.getNativeModel()
    import glob
    import tempfile
    pasta = tempfile.mkdtemp()
    ajustado.saveNativeModel(f"file:{pasta}/modelo", overwrite=True)
    arquivos = sorted(glob.glob(f"{pasta}/modelo/part-*")) or sorted(glob.glob(f"{pasta}/modelo*"))
    return "".join(open(caminho, encoding="utf-8").read() for caminho in arquivos)


def espelho(parte, X):
    """
    Espelho em pandas de uma parte do modelo.

    Devolve dict com: shap (n × atributos), valor_base, previsao (escala do
    score), importancia_nativa (por atributo), escala_shap.
    """
    tecnica, tarefa, ajustado = parte["tecnica"], parte["tarefa"], parte["ajustado"]
    matriz = X.to_numpy(dtype=float)

    if tecnica in ("logistica", "logistica_fracionaria"):
        beta = np.asarray(ajustado.coefficients.toArray())
        media = matriz.mean(axis=0)
        margem = ajustado.intercept + matriz @ beta
        return {"shap": (matriz - media) * beta, "valor_base": float(ajustado.intercept + media @ beta),
                "previsao": 1 / (1 + np.exp(-margem)),
                "importancia_nativa": np.abs(beta) * matriz.std(axis=0), "escala_shap": "logit"}

    if tecnica in ("random_forest", "gbt"):
        import shap
        explicador = shap.TreeExplainer(ajustado)
        valores = np.asarray(explicador.shap_values(X, check_additivity=False))
        base = np.atleast_1d(explicador.expected_value)
        if valores.ndim == 3:                         # classificação do Random Forest: classe 1
            valores, base = valores[:, :, 1], base[1]
        else:
            base = base[-1]
        margem = base + valores.sum(axis=1)
        if tecnica == "gbt" and tarefa == "pd":
            # O GBT do Spark usa rótulos ±1: probabilidade = 1 / (1 + e^(−2·margem)).
            previsao, escala = 1 / (1 + np.exp(-2 * margem)), "margem do GBT"
        elif tarefa == "pd":
            previsao, escala = margem, "probabilidade"
        else:
            # Regressão (LGD): a escoragem Spark limita a previsão a [0, 1]
            # (_escora_parte, notebook 03); o espelho aplica o mesmo limite. O
            # SHAP continua na escala antes do limite, onde a soma é exata.
            previsao, escala = np.clip(margem, 0.0, 1.0), "LGD (antes do limite [0, 1])"
        return {"shap": valores, "valor_base": float(base), "previsao": previsao,
                "importancia_nativa": np.asarray(ajustado.featureImportances.toArray()), "escala_shap": escala}

    if tecnica == "xgboost":
        import xgboost as xgb
        booster = ajustado.get_booster()
        dados = xgb.DMatrix(matriz, missing=np.nan, feature_names=booster.feature_names)
        contribuicoes = booster.predict(dados, pred_contribs=True)
        nomes = booster.feature_names or [f"f{i}" for i in range(matriz.shape[1])]
        ganho = booster.get_score(importance_type="gain")
        return {"shap": contribuicoes[:, :-1], "valor_base": float(contribuicoes[0, -1]),
                "previsao": booster.predict(dados),
                "importancia_nativa": np.array([ganho.get(nome, 0.0) for nome in nomes]),
                "escala_shap": "logit" if tarefa == "pd" else "margem (logit da LGD)"}

    if tecnica == "lightgbm":
        import lightgbm as lgb
        booster = lgb.Booster(model_str=_texto_modelo_lightgbm(ajustado))
        contribuicoes = booster.predict(matriz, pred_contrib=True)
        return {"shap": contribuicoes[:, :-1], "valor_base": float(contribuicoes[0, -1]),
                "previsao": booster.predict(matriz),
                "importancia_nativa": booster.feature_importance(importance_type="gain").astype(float),
                "escala_shap": "logit" if tarefa == "pd" else "margem (logit da LGD)"}

    raise ValueError(f"técnica sem espelho: {tecnica}")

## 5. De atributos para variáveis, e o teste de fidelidade

Um atributo não é sempre uma variável: uma categórica em one-hot vira várias colunas, e uma numérica com vazios ganha o indicador de faltante. O SHAP e a importância de uma variável são a **soma** dos seus atributos (o SHAP é aditivo, então a soma é exata).

Para colorir o beeswarm, cada variável precisa de um "valor": numéricas usam o valor original; categóricas usam a posição da categoria na ordem de risco (0 = menor risco), de modo que vermelho = categoria de risco mais alto.

**Fidelidade** (na amostra):

| medida | aprovação |
|---|---|
| Spearman entre o score do Spark e o do espelho | ≥ 0,95 |
| % de contratos no mesmo decil do score | ≥ 90% |
| diferença de KS (PD) ou de RMSE (LGD) | informativa |
| maior diferença absoluta entre os scores | informativa: perto de 0 = espelho exato |

In [ ]:
def agrega_por_variavel(parte, pdf, resultado_espelho, contexto):
    """SHAP, valores e importância por variável original (pandas)."""
    atributos = parte["plano"]["atributos"]
    shap_var, valor_var, importancia_nativa = {}, {}, {}
    for variavel in contexto["variaveis"]:
        indices = [i for i, a in enumerate(atributos) if a["variavel"] == variavel]
        if not indices:
            continue
        shap_var[variavel] = resultado_espelho["shap"][:, indices].sum(axis=1)
        importancia_nativa[variavel] = float(resultado_espelho["importancia_nativa"][indices].sum())
        original = pdf[f"orig__{variavel}"]
        if contexto["tipos"][variavel] == "categorica":
            ordem = parte["plano"]["ordem_categorias"][variavel]
            valor_var[variavel] = original.map({c: i for i, c in enumerate(ordem)}).astype(float)
        else:
            valor_var[variavel] = pd.to_numeric(original, errors="coerce").astype(float)
    shap_var, valor_var = pd.DataFrame(shap_var), pd.DataFrame(valor_var)
    total_nativa = sum(importancia_nativa.values()) or 1.0
    importancia = pd.DataFrame({
        "variavel": list(shap_var.columns),
        "importancia_nativa_pct": [100 * importancia_nativa[v] / total_nativa for v in shap_var.columns],
        "shap_medio_abs": shap_var.abs().mean().values,
    }).sort_values("shap_medio_abs", ascending=False).reset_index(drop=True)
    return shap_var, valor_var, importancia


def _ks(y, score):
    """KS entre bons e maus (numpy)."""
    ordem = np.argsort(score)
    y = np.asarray(y)[ordem]
    acum_maus = np.cumsum(y) / max(y.sum(), 1)
    acum_bons = np.cumsum(1 - y) / max((1 - y).sum(), 1)
    return float(np.max(np.abs(acum_maus - acum_bons)))


def teste_fidelidade(nome_parte, parte, pdf, resultado_espelho, cfg):
    """Uma linha comparando o score do Spark e o do espelho na amostra."""
    from scipy.stats import spearmanr
    spark_score = pdf["_score_spark"].to_numpy(float)
    espelho_score = np.asarray(resultado_espelho["previsao"], dtype=float)
    decil_spark = pd.qcut(pd.Series(spark_score).rank(method="first"), 10, labels=False)
    decil_espelho = pd.qcut(pd.Series(espelho_score).rank(method="first"), 10, labels=False)
    y = pdf[cfg["colunas"]["alvo"]].to_numpy(float)
    if parte["tarefa"] == "pd" and cfg["tarefa"] == "pd":
        metrica, spark_m, espelho_m = "ks", _ks(y, spark_score), _ks(y, espelho_score)
    elif parte["tarefa"] == "pd":                     # etapa 1 do duas etapas: alvo = perda ≥ limiar
        rotulo = (y >= cfg["automl"]["limiar_duas_etapas"]).astype(float)
        metrica, spark_m, espelho_m = "ks", _ks(rotulo, spark_score), _ks(rotulo, espelho_score)
    else:
        metrica = "rmse"
        spark_m = float(np.sqrt(np.mean((spark_score - y) ** 2)))
        espelho_m = float(np.sqrt(np.mean((np.clip(espelho_score, 0, 1) - y) ** 2)))
    e = cfg["explicabilidade"]
    linha = {"parte": nome_parte, "tecnica": parte["tecnica"], "n_amostra": len(pdf),
             "spearman": float(spearmanr(spark_score, espelho_score)[0]),
             "pct_mesmo_decil": float((decil_spark == decil_espelho).mean()),
             "metrica": metrica, "valor_spark": spark_m, "valor_espelho": espelho_m,
             "maior_diferenca_absoluta": float(np.max(np.abs(spark_score - espelho_score)))}
    linha["aprovado"] = (linha["spearman"] >= e["fidelidade_min_spearman"]
                         and linha["pct_mesmo_decil"] >= e["fidelidade_min_mesmo_rating"])
    linha["espelho_exato"] = linha["maior_diferenca_absoluta"] < 1e-4
    return linha

## 6. Gráficos: beeswarm e importância

**Beeswarm.** Uma linha por variável (as mais importantes no topo); cada ponto é um contrato. Posição horizontal = quanto aquela variável empurrou a previsão daquele contrato (direita aumenta o risco); cor = valor da variável (vermelho alto, azul baixo). Desenha até `n_pontos_beeswarm` contratos, para o gráfico continuar legível.

**Importância.** Dois painéis com as mesmas variáveis, na ordem do SHAP: à esquerda a importância nativa do modelo (% do total), à direita o SHAP médio em valor absoluto. As duas medidas podem discordar: o ganho favorece variáveis contínuas com muitos cortes; o SHAP mede o efeito na previsão de cada contrato.

In [ ]:
def grafico_beeswarm(shap_var, valor_var, cfg, titulo, escala):
    """Beeswarm do SHAP por variável."""
    import matplotlib.pyplot as plt
    import shap
    n_pontos = min(cfg["explicabilidade"]["n_pontos_beeswarm"], len(shap_var))
    indices = np.random.default_rng(cfg["semente"]).choice(len(shap_var), size=n_pontos, replace=False)
    plt.figure()
    shap.summary_plot(shap_var.to_numpy()[indices], valor_var.iloc[indices], show=False,
                      max_display=cfg["explicabilidade"]["n_variaveis_grafico"],
                      plot_size=(9, max(3.0, 0.42 * min(len(shap_var.columns),
                                                        cfg["explicabilidade"]["n_variaveis_grafico"]) + 1.6)),
                      color_bar_label="valor da variável")
    figura = plt.gcf()
    eixo = figura.axes[0]
    eixo.set_xlabel(f"valor SHAP (escala: {escala}); à direita aumenta o risco")
    eixo.set_title(titulo, loc="left", fontsize=11)
    if len(figura.axes) > 1:
        figura.axes[-1].set_yticklabels(["baixo", "alto"])
    return figura


def grafico_importancia(importancia, cfg, titulo, escala):
    """Dois painéis: importância nativa (%) e SHAP médio absoluto, mesma ordem."""
    import matplotlib.pyplot as plt
    dados = importancia.head(cfg["explicabilidade"]["n_variaveis_grafico"]).iloc[::-1]
    altura = max(3.0, 0.32 * len(dados) + 1.4)
    figura, (esquerda, direita) = plt.subplots(1, 2, figsize=(11, altura), dpi=130, sharey=True)
    esquerda.barh(dados["variavel"], dados["importancia_nativa_pct"], color="#2a78d6", height=0.6)
    direita.barh(dados["variavel"], dados["shap_medio_abs"], color="#eb6834", height=0.6)
    for eixo, coluna, formato in ((esquerda, "importancia_nativa_pct", "{:.0f}%"), (direita, "shap_medio_abs", "{:.3f}")):
        maximo = dados[coluna].max() or 1.0
        for i, valor in enumerate(dados[coluna]):
            eixo.text(valor + 0.01 * maximo, i, formato.format(valor).replace(".", ","),
                      va="center", fontsize=8, color="#52514e")
        eixo.set_xlim(0, maximo * 1.2)
        eixo.spines[["top", "right"]].set_visible(False)
        eixo.grid(axis="x", color="#e4e3df", lw=0.8)
        eixo.set_axisbelow(True)
        eixo.tick_params(labelsize=8)
    esquerda.set_title("importância nativa do modelo (% do total)", loc="left", fontsize=10)
    direita.set_title(f"|SHAP| médio (escala: {escala})", loc="left", fontsize=10)
    figura.suptitle(titulo, x=0.01, ha="left", fontsize=11)
    figura.tight_layout()
    return figura

## 7. Execução das etapas 5 e 6

In [ ]:
def executa_etapa_calibracao_explicabilidade(spark, etapa_automl, etapa_selecao, cfg, gravar=True):
    """
    Etapas 5 (calibração) e 6 (explicabilidade).

    Devolve
        base_calibrada      Spark, com score_bruto e score_calibrado (todas as partições)
        calibracao          parâmetros da calibração
        tabela_calibracao   pandas, antes × depois por partição
        importancia         {parte: pandas por variável}
        fidelidade          pandas, uma linha por parte
        figura_calibracao, figuras_beeswarm, figuras_importancia
    """
    modelo, contexto = etapa_automl["modelo"], etapa_automl["contexto"]

    # ---- etapa 5: calibração ------------------------------------------------
    calibracao = ajusta_calibracao(etapa_automl["base_escorada"], cfg)
    print(f"[calibração] {calibracao['metodo']}: "
          + (", ".join(f"{k}={v:.4f}" for k, v in calibracao.items() if isinstance(v, float))
             or f"{len(calibracao.get('pontos', []))} pontos"))
    base_calibrada = aplica_calibracao(etapa_automl["base_escorada"], calibracao).persist()
    tabela_cal = tabela_calibracao(base_calibrada, cfg)
    exibe(tabela_cal, "Calibração: antes × depois")
    figura_cal = grafico_calibracao(base_calibrada, cfg)
    registra_figura_mlflow(figura_cal, "calibracao/oot.png")

    # ---- etapa 6: espelho, SHAP, fidelidade e gráficos ----------------------
    amostras = amostra_para_explicacao(base_calibrada, modelo, contexto, cfg)
    importancias, fidelidades, beeswarms, graficos_importancia = {}, [], {}, {}
    rotulos = {"modelo": "", "etapa1": " — etapa 1: chance de perda", "etapa2": " — etapa 2: severidade"}
    for nome, (parte, pdf) in amostras.items():
        colunas_f = [a["coluna"] for a in parte["plano"]["atributos"]]
        resultado = espelho(parte, pdf[colunas_f])
        shap_var, valor_var, importancia = agrega_por_variavel(parte, pdf, resultado, contexto)
        importancias[nome] = importancia
        fidelidades.append(teste_fidelidade(nome, parte, pdf, resultado, cfg))
        titulo = f"{parte['tecnica']}{rotulos[nome]}"
        beeswarms[nome] = grafico_beeswarm(shap_var, valor_var, cfg, f"SHAP por contrato: {titulo}",
                                           resultado["escala_shap"])
        graficos_importancia[nome] = grafico_importancia(importancia, cfg, f"Importância das variáveis: {titulo}",
                                                         resultado["escala_shap"])
        registra_figura_mlflow(beeswarms[nome], f"explicabilidade/beeswarm_{nome}.png")
        registra_figura_mlflow(graficos_importancia[nome], f"explicabilidade/importancia_{nome}.png")
        exibe(importancia, f"Importância por variável ({nome})")

    fidelidade = pd.DataFrame(fidelidades)
    exibe(fidelidade, "Fidelidade do espelho")
    if not fidelidade["aprovado"].all():
        print("[explicabilidade] ATENÇÃO: espelho reprovado no teste de fidelidade; "
              "a explicação é aproximada (ver tabela).")

    if gravar:
        grava_tabela_delta(spark, tabela_cal, "calibracao", cfg)
        grava_tabela_delta(spark, pd.DataFrame([{"parametros": json.dumps(calibracao)}]), "calibracao_parametros", cfg)
        grava_tabela_delta(spark, fidelidade, "explicabilidade_fidelidade", cfg)
        for nome, importancia in importancias.items():
            grava_tabela_delta(spark, importancia.assign(parte=nome), "explicabilidade_importancia", cfg)

    return {"base_calibrada": base_calibrada, "calibracao": calibracao, "tabela_calibracao": tabela_cal,
            "importancia": importancias, "fidelidade": fidelidade, "figura_calibracao": figura_cal,
            "figuras_beeswarm": beeswarms, "figuras_importancia": graficos_importancia}

## 8. Exemplo de uso

```python
etapa56 = executa_etapa_calibracao_explicabilidade(spark, etapa4, etapa2, cfg)
etapa56["fidelidade"]                       # espelho × Spark
etapa56["figuras_beeswarm"]["modelo"]       # beeswarm
etapa56["base_calibrada"]                   # segue para a régua (notebook 05)
```